# Proposed Data Cleaning

This notebook will explore filtering datasets to make analysis simpler. MARKED FOR REVISION this notebook is basically not even started yet, don't run the code below 

In [ ]:
WR_PLAYER_PLAY_COLUMNS = [
    # ------------------------------------------
    # A. Identifiers and game context
    # ------------------------------------------
    "game_id",
    "play_id",
    "team_abbr",
    "nfl_id",
    "lined_up_position",
    "play_description",
    "quarter",
    "down",
    "yards_to_go",
    "possession_team",
    "yardline_side",
    "yardline_number",
    "game_clock",
    "pre_snap_home_score",
    "pre_snap_visitor_score",
    "play_direction",

    # ------------------------------------------
    # B. Win probability and expected points
    # ------------------------------------------
    "pre_snap_home_team_win_probability",
    "pre_snap_visitor_team_win_probability",
    "home_team_win_probability_added",
    "visitor_team_win_probility_added",
    "expected_points_added",
    "expected_points",

    # ------------------------------------------
    # C. Formations and play outcomes
    # ------------------------------------------
    "offense_formation",
    "receiver_alignment",
    "pass_result",
    "pass_length",
    "play_nullified_by_penalty",
    "penalty_yards",
    "pre_penalty_yards_gained",
    "yards_gained",
    "team_coverage_man_zone",
    "team_coverage_type",
    "in_motion_at_ball_snap",

    # ------------------------------------------
    # D. WR and ball-carrier metrics
    # ------------------------------------------
    "target",
    "rec_yards",
    "rush_yards",
    "yards_after_catch",
    "fumble",
    "fumble_lost",
    "route_ran",
    "separation_at_pass_forward",
    "cushion",
    "expected_yards_after_catch",

    # ------------------------------------------
    # E. Relevant QB context
    # ------------------------------------------
    "dropback_duration",
]

In [ ]:
# Identify WRs using the players table
wr_ids = set(
    players.loc[
        players["nfl_position"] == "WR",
        "nfl_id"
    ]
)

# Keep only WR observations
player_play_wr = player_play.loc[
    player_play["nfl_id"].isin(wr_ids)
].copy()

# Select documented WR-relevant columns
available_cols = [
    col for col in WR_PLAYER_PLAY_COLUMNS
    if col in player_play_wr.columns
]

missing_cols = sorted(
    set(WR_PLAYER_PLAY_COLUMNS) - set(available_cols)
)

player_play_wr = player_play_wr[available_cols].copy()

print(f"WR observations: {len(player_play_wr):,}")
print(f"Unique WRs: {player_play_wr['nfl_id'].nunique():,}")
print(f"Columns retained: {len(available_cols)}")

if missing_cols:
    print("Missing requested columns:", missing_cols)

display(player_play_wr.head())

In [ ]:
# 1. Ensure all records belong to identified WRs
assert player_play_wr["nfl_id"].isin(wr_ids).all()

# 2. Check uniqueness of player-play observations
duplicate_count = player_play_wr.duplicated(
    subset=["game_id", "play_id", "nfl_id"]
).sum()

print(f"Duplicate player-play keys: {duplicate_count:,}")

# 3. Inspect missingness on important WR variables
important_cols = [
    "route_ran",
    "target",
    "separation_at_pass_forward",
    "cushion",
    "yards_after_catch",
    "team_coverage_type",
    "dropback_duration",
]

missingness = (
    player_play_wr[
        [c for c in important_cols if c in player_play_wr.columns]
    ]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

display(missingness.to_frame("Missing (%)"))